## Fase 1: Comprensión del Negocio

### Empresa: HealthGuard Analytics

Una empresa líder en soluciones tecnológicas para el sector sanitario, especializada en el desarrollo de sistemas predictivos para la detección temprana de enfermedades crónicas.

### Misión
Transformar la atención médica preventiva mediante el uso de tecnologías avanzadas de análisis de datos, mejorando la calidad de vida de las personas a través de la detección temprana de condiciones de salud.
### Visión
Ser el referente global en sistemas de predicción de enfermedades crónicas, estableciendo nuevos estándares en la medicina preventiva personalizada mediante la integración de inteligencia artificial y análisis de datos clínicos.

### Objetivo Comercial
Incrementar la eficiencia en la detección temprana de la diabetes para lograr un ahorro del 30% en los costos asociados en un año.

__KPI:__ Porcentaje de reducción de costos: Comparación de costos operativos antes y después de implementar la solución.

### Objetivo de Minería de Datos
Identificar a aquellos con un riesgo elevado de desarrollar diabetes, priorizando la detección temprana y facilitando la intervención médica preventiva.

__KPI:__ Cantidad de pacientes analizados: Número total de registros procesados dentro del período establecido.

## Fase 2: Comprensión de los Datos

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

In [10]:
df = pd.read_csv('diabetes_prediction_dataset.csv')
df

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0
...,...,...,...,...,...,...,...,...,...
99995,Female,80.0,0,0,No Info,27.32,6.2,90,0
99996,Female,2.0,0,0,No Info,17.37,6.5,100,0
99997,Male,66.0,0,0,former,27.83,5.7,155,0
99998,Female,24.0,0,0,never,35.42,4.0,100,0


In [16]:
df.shape

(100000, 9)

El dataset contiene 100,000 registros y 9 columnas. 
- gender: Categórico nominal (objeto)
- age: Numérico (float64)
- hypertension: Binario (0 o 1, int64)
- heart_disease: Binario (0 o 1, int64)
- smoking_history: Categórico nominal (objeto)
- bmi: Numérico continuo (float64)
- HbA1c_level: Numérico continuo (float64)
- blood_glucose_level: Numérico discreto (int64)
- diabetes: Binario (0 o 1, int64), es la variable objetivo.

In [11]:
# Distribución de la variable objetivo
print(df['diabetes'].value_counts(normalize=True) * 100)  # Porcentajes
print(df['diabetes'].value_counts())  # Cantidad absoluta

diabetes
0    91.5
1     8.5
Name: proportion, dtype: float64
diabetes
0    91500
1     8500
Name: count, dtype: int64


In [13]:
# Distribución de la variable género
print(df['gender'].value_counts(normalize=True) * 100)  # Porcentajes
print(df['gender'].value_counts())  # Cantidad absoluta

gender
Female    58.552
Male      41.430
Other      0.018
Name: proportion, dtype: float64
gender
Female    58552
Male      41430
Other        18
Name: count, dtype: int64


In [12]:
# Revisión de valores nulos
print(df.isnull().sum())

gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64


In [14]:
df.describe()

,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,diabetes
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,41.885856,0.07485,0.039420,27.320767,5.527507,138.058060,0.085000
std,22.516840,0.26315,0.194593,6.636783,1.070672,40.708136,0.278883
min,0.080000,0.00000,0.000000,10.010000,3.500000,80.000000,0.000000
25%,24.000000,0.00000,0.000000,23.630000,4.800000,100.000000,0.000000
50%,43.000000,0.00000,0.000000,27.320000,5.800000,140.000000,0.000000
75%,60.000000,0.00000,0.000000,29.580000,6.200000,159.000000,0.000000
max,80.000000,1.00000,1.000000,95.690000,9.000000,300.000000,1.000000


## Fase 3: Preparación de los Datos

In [17]:
# Manejo de duplicados
duplicate_rows_data = df[df.duplicated()]
print("Number of duplicate rows: ", duplicate_rows_data.shape)

Number of duplicate rows:  (3854, 9)


In [18]:
df = df.drop_duplicates()

In [26]:
# Remover el valor innecesario de 'Other' dentro de género debido a que representa un valor de 0.018%.
df = df[df['gender'] != 'Other']

In [21]:
# Reasignar categorías usando replace
df['smoking_history'] = df['smoking_history'].replace({
    'never': 'non-smoker',
    'No Info': 'non-smoker',
    'current': 'current',
    'ever': 'past_smoker',
    'former': 'past_smoker',
    'not current': 'past_smoker'
})

# Comprobar los nuevos valores
print(df['smoking_history'].value_counts())

smoking_history
non-smoker     67276
past_smoker    19655
current         9197
Name: count, dtype: int64


In [24]:
# Preprocessing: Encoding categorical variables and scaling numeric ones
df['gender'] = LabelEncoder().fit_transform(df['gender'])
df['smoking_history'] = LabelEncoder().fit_transform(df['smoking_history'])

In [25]:
df

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,0,80.0,0,1,1,25.19,6.6,140,0
1,0,54.0,0,0,1,27.32,6.6,80,0
2,1,28.0,0,0,1,27.32,5.7,158,0
3,0,36.0,0,0,0,23.45,5.0,155,0
4,1,76.0,1,1,0,20.14,4.8,155,0
...,...,...,...,...,...,...,...,...,...
99994,0,36.0,0,0,1,24.60,4.8,145,0
99996,0,2.0,0,0,1,17.37,6.5,100,0
99997,1,66.0,0,0,2,27.83,5.7,155,0
99998,0,24.0,0,0,1,35.42,4.0,100,0


In [28]:
# Separating features and target variable
X = df.drop('diabetes', axis=1)
y = df['diabetes']

## Fase 4: Modelado

In [34]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to balance the dataset
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Split the resampled data into train and test sets
X_train_resampled, X_test_resampled, y_train_resampled, y_test_resampled = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Scaling numerical features
scaler = StandardScaler()
# Scale the resampled data
X_train_resampled_scaled = scaler.fit_transform(X_train_resampled)
X_test_resampled_scaled = scaler.transform(X_test_resampled)

# Models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "Gradient Boost": GradientBoostingClassifier(random_state=42)
}

# Retrain models on the balanced dataset
balanced_results = {}
for name, model in models.items():
    model.fit(X_train_resampled_scaled, y_train_resampled)
    y_pred_resampled = model.predict(X_test_resampled_scaled)
    balanced_results[name] = classification_report(y_test_resampled, y_pred_resampled, output_dict=True)

balanced_results

{'Random Forest': {'0': {'precision': 0.9694335549116527,
   'recall': 0.9733599543639475,
   'f1-score': 0.9713927869972389,
   'support': 17530.0},
  '1': {'precision': 0.9732500859204949,
   'recall': 0.9693080038792857,
   'f1-score': 0.9712750450167205,
   'support': 17529.0},
  'accuracy': 0.9713340369092102,
  'macro avg': {'precision': 0.9713418204160738,
   'recall': 0.9713339791216167,
   'f1-score': 0.9713339160069796,
   'support': 35059.0},
  'weighted avg': {'precision': 0.9713417659859559,
   'recall': 0.9713340369092102,
   'f1-score': 0.9713339176861773,
   'support': 35059.0}},
 'SVM': {'0': {'precision': 0.9259705085765875,
   'recall': 0.8776383342840844,
   'f1-score': 0.9011568311612241,
   'support': 17530.0},
  '1': {'precision': 0.8837020169160703,
   'recall': 0.9298305664898169,
   'f1-score': 0.9061796347260446,
   'support': 17529.0},
  'accuracy': 0.9037337060383924,
  'macro avg': {'precision': 0.9048362627463289,
   'recall': 0.9037344503869507,
   'f1-s

## Fase 5: Evaluación

Basándonos en los resultados obtenidos en la fase de modelado, procederemos a realizar una evaluación detallada de los tres modelos implementados para la predicción de diabetes.


El modelo Random Forest ha demostrado un rendimiento excepcional, alcanzando una precisión global del 97.13%. Su capacidad para identificar tanto casos positivos como negativos es notablemente equilibrada, con una precisión del 96.94% para pacientes no diabéticos y 97.33% para pacientes diabéticos. La consistencia en el recall (97.34% para no diabéticos y 96.93% para diabéticos) indica que el modelo es igualmente efectivo en la identificación de ambas clases.


El modelo SVM, aunque con un rendimiento inferior, mantiene una precisión global respetable del 90.37%. Su comportamiento muestra cierta asimetría en la clasificación, con una precisión más alta para pacientes no diabéticos (92.60%) que para diabéticos (88.37%). Esta diferencia sugiere que el modelo podría beneficiarse de un ajuste adicional en sus hiperparámetros para mejorar el equilibrio entre clases.


El Gradient Boosting presenta un rendimiento muy sólido con una precisión global del 96.27%. Este modelo muestra una interesante complementariedad en sus métricas: mientras que obtiene una precisión más alta para pacientes diabéticos (98.33%), su recall es superior para pacientes no diabéticos (98.40%). Este patrón indica que el modelo es particularmente confiable cuando identifica casos positivos de diabetes, aunque puede ser ligeramente más conservador en sus predicciones.

Comparando los tres modelos, Random Forest emerge como la opción más robusta para su implementación en producción por las siguientes razones:

Presenta el mejor rendimiento global con una precisión del 97.13%.
Mantiene el mejor equilibrio entre precisión y recall para ambas clases.
Demuestra la mayor consistencia en todas las métricas de evaluación.

Esta evaluación sugiere que el modelo Random Forest cumple satisfactoriamente con los objetivos comerciales de HealthGuard Analytics, superando el umbral de precisión del 85% establecido inicialmente. Su capacidad para mantener un rendimiento equilibrado entre las diferentes clases lo hace particularmente valioso para aplicaciones clínicas, donde tanto los falsos positivos como los falsos negativos pueden tener implicaciones significativas para la atención al paciente.